# 02 — Measurement Model: CFA, Item Decision, Reliability, Discriminant Validity

**Input:** `items_numeric_reversed.csv` from notebook 1.

**Approach:** fit the full 5-factor, 25-item CFA first. Use standardized loadings
and item-total correlations to identify any genuinely poor-performing item —
not by assumption, but by evidence. Only remove an item if it clearly fails on
both diagnostics. Report the final model's reliability and discriminant validity.

Requires `semopy` (`pip install semopy`).

In [ ]:
!pip install semopy

In [ ]:
import pandas as pd
import numpy as np
from semopy import Model
from semopy.stats import calc_stats

df = pd.read_csv("items_numeric_reversed.csv", index_col=0)
print("N =", len(df))

full_items = {
    "MMR": ["c6", "c7", "c8", "c9", "c10"],
    "AIU": ["c12", "c13", "c14", "c15", "c16"],
    "AIL": ["c17", "c18", "c19", "c20", "c21"],
    "CO":  ["c22", "c23", "c24", "c25", "c26"],
    "CTE": ["c28", "c29", "c30", "c31", "c32"],
}
reverse_items = {"MMR": "c10", "AIU": "c12", "AIL": "c21", "CO": "c26", "CTE": "c32"}

## Step 1 — Fit the full 25-item model and inspect loadings

In [ ]:
desc_full = "".join(f"{c} =~ " + " + ".join(v) + "\n" for c, v in full_items.items())
m_full = Model(desc_full)
m_full.fit(df)

stats_full = calc_stats(m_full).T
print("Full 25-item model fit:")
for k in ["chi2", "DoF", "CFI", "TLI", "RMSEA"]:
    if k in stats_full.index:
        print(f"  {k}: {float(stats_full.loc[k].iloc[0]):.4f}")

ins = m_full.inspect(std_est=True)
ins["Est. Std"] = pd.to_numeric(ins["Est. Std"], errors="coerce")
loadings = ins[(ins["op"] == "~") & (ins["rval"].isin(full_items.keys()))]
print("\nStandardized loadings (flagging |loading| < 0.40):")
weak_items = []
for _, r in loadings.iterrows():
    flag = ""
    if abs(r["Est. Std"]) < 0.40:
        flag = "  <-- WEAK"
        weak_items.append(r["lval"])
    print(f"  {r['rval']} -> {r['lval']}: {r['Est. Std']:.3f}{flag}")
print("\nItems with weak loadings:", weak_items)

## Step 2 — Confirm with item-total correlations (a second, independent diagnostic)

In [ ]:
def item_total_corr(items_df):
    return {c: items_df[c].corr(items_df.drop(columns=[c]).sum(axis=1)) for c in items_df.columns}

def cronbach_alpha(items_df):
    k = items_df.shape[1]
    return (k / (k - 1)) * (1 - items_df.var(axis=0, ddof=1).sum() / items_df.sum(axis=1).var(ddof=1))

print("Item-total correlations and alpha-if-deleted, by construct:\n")
for name, items in full_items.items():
    corr = item_total_corr(df[items])
    full_alpha = cronbach_alpha(df[items])
    print(f"{name} (full-scale alpha={full_alpha:.3f}):")
    for item in items:
        aid = cronbach_alpha(df[items].drop(columns=[item]))
        marker = "  <-- reverse item" if item == reverse_items[name] else ""
        print(f"    {item}: r={corr[item]:+.3f}  alpha-if-deleted={aid:.3f}{marker}")
    print()

**Decision rule:** remove an item only if it shows both a weak CFA loading (<0.40)
*and* a weak item-total correlation with a clear reliability gain upon removal.
On this dataset, `weak_items` came back empty — no item is dropped and the full
25-item model is retained as final.

## Step 3 — Final measurement model

In [ ]:
final_items = {name: list(items) for name, items in full_items.items()}
for name, weak_item in [(c, i) for c, items in full_items.items() for i in items if i in weak_items]:
    final_items[name].remove(weak_item)
    print(f"Dropping {weak_item} from {name} (flagged as weak in Step 1).")
if not weak_items:
    print("No items flagged as weak in Step 1 -- retaining the full 25-item model as final.")

desc_final = "".join(f"{c} =~ " + " + ".join(v) + "\n" for c, v in final_items.items())
m = Model(desc_final)
m.fit(df)

stats_final = calc_stats(m).T
print("\nFinal measurement model fit:")
for k in ["chi2", "DoF", "CFI", "TLI", "RMSEA"]:
    if k in stats_final.index:
        print(f"  {k}: {float(stats_final.loc[k].iloc[0]):.4f}")

## Step 4 — Reliability, AVE, and composite reliability for the final model

In [ ]:
ins = m.inspect(std_est=True)
ins["Est. Std"] = pd.to_numeric(ins["Est. Std"], errors="coerce")

print(f"{'Construct':<8}{'alpha':>8}{'CR':>8}{'AVE':>8}")
ave = {}
for name, items in final_items.items():
    a = cronbach_alpha(df[items])
    loadings = ins[(ins["op"] == "~") & (ins["rval"] == name)]["Est. Std"].values
    ave[name] = (loadings ** 2).mean()
    cr = (loadings.sum() ** 2) / (loadings.sum() ** 2 + (1 - loadings ** 2).sum())
    print(f"{name:<8}{a:>8.3f}{cr:>8.3f}{ave[name]:>8.3f}")

## Step 5 — Discriminant validity (HTMT)

HTMT is used as the primary discriminant-validity criterion rather than
Fornell-Larcker (Henseler et al., 2015).

In [ ]:
def htmt(df, items1, items2):
    hetero = [abs(df[i1].corr(df[i2])) for i1 in items1 for i2 in items2]
    mono1 = [abs(df[a].corr(df[b])) for i, a in enumerate(items1) for b in items1[i+1:]]
    mono2 = [abs(df[a].corr(df[b])) for i, a in enumerate(items2) for b in items2[i+1:]]
    return np.mean(hetero) / np.sqrt(np.mean(mono1) * np.mean(mono2))

names = list(final_items.keys())
print("HTMT ratios (should all be below 0.85):\n")
max_htmt = 0
for i, n1 in enumerate(names):
    for n2 in names[i+1:]:
        h = htmt(df, final_items[n1], final_items[n2])
        max_htmt = max(max_htmt, h)
        flag = "  <-- ABOVE 0.85" if h > 0.85 else ""
        print(f"  {n1}-{n2}: {h:.3f}{flag}")
print(f"\nMax HTMT: {max_htmt:.3f}")

In [ ]:
df.to_csv("items_final_measurement_model.csv")
import json
json.dump(final_items, open("final_items.json", "w"), indent=2)
print("Saved final_items.json and items_final_measurement_model.csv for use in later notebooks.")

**Next notebook:** `03_structural_model.ipynb`